# Session 08 Lab: Neural Networks
**Course:** Noob to AI Expert | **Track:** Intermediate

Build and train a digit classifier with PyTorch. Understand neurons, layers, backpropagation, and overfitting.

In [ ]:
!pip install torch torchvision matplotlib -q
print('✅ Ready')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

transform = transforms.ToTensor()
train_data = datasets.MNIST('.', train=True, download=True, transform=transform)
test_data = datasets.MNIST('.', train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

print(f'Train: {len(train_data)} | Test: {len(test_data)}')

In [ ]:
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),          # 28x28 → 784
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(0.3),       # Regularization
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10)     # 10 digit classes
        )

    def forward(self, x):
        return self.net(x)

model = DigitClassifier()
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
train_losses, test_accs = [], []

for epoch in range(5):
    model.train()
    epoch_loss = 0
    for X, y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))

    model.eval()
    correct = 0
    with torch.no_grad():
        for X, y in test_loader:
            correct += (model(X).argmax(1) == y).sum().item()
    acc = correct / len(test_data)
    test_accs.append(acc)
    print(f'Epoch {epoch+1}: loss={train_losses[-1]:.4f}, test_acc={acc:.4f}')

In [ ]:
# Visualize predictions
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(images).argmax(1)

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    color = 'green' if preds[i] == labels[i] else 'red'
    ax.set_title(f'Pred: {preds[i].item()}', color=color)
    ax.axis('off')
plt.suptitle('Green=Correct, Red=Wrong')
plt.tight_layout()
plt.show()